# 00_config / 03__tickers_master

Builds the unified ticker universe by merging:
- **S&P 500** — 500 large-cap US stocks (Wikipedia)
- **Russell 3000** — broad US market ~3000 stocks (iShares IWV ETF)
- **Favorites** — from `main.config.favorites` (managed in `02__favorites`)

**Output table:** `main.config.tickers`

```
ticker | company    | in_sp500 | in_r3000 | is_favorite
AAPL   | Apple      | true     | true     | false
TSM    | TSMC       | false    | false    | true
```

In [0]:
%run "./01__tickers"

In [0]:
import requests
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

INGEST_SP500  = True
INGEST_R3000  = True

TARGET_TABLE    = f"{CATALOG}.config.tickers"
FAVORITES_TABLE = f"{CATALOG}.config.favorites"

_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}

## 1. Pull index constituents + favorites

In [0]:
def fetch_sp500() -> pd.DataFrame:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    html = requests.get(url, headers=_HEADERS).text
    df = pd.read_html(html, flavor="lxml")[0][["Symbol", "Security"]].copy()
    df.columns = ["ticker", "company"]
    df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)
    return df.dropna(subset=["ticker"])


def fetch_russell3000() -> pd.DataFrame:
    url = (
        "https://www.ishares.com/us/products/239714/ishares-russell-3000-etf"
        "/1467271812596.ajax?fileType=csv&fileName=IWV_holdings&dataType=fund"
    )
    resp = requests.get(url, headers=_HEADERS)
    df = pd.read_csv(StringIO(resp.text), skiprows=9, header=0)
    df = df[["Ticker", "Name"]].dropna(subset=["Ticker"])
    df.columns = ["ticker", "company"]
    df = df[df["ticker"].str.match(r"^[A-Z\-]+$")]
    return df.reset_index(drop=True)


def fetch_favorites() -> pd.DataFrame:
    try:
        df = spark.table(FAVORITES_TABLE).select("ticker", "company").toPandas()
        print(f"  ✓ {len(df)} favorite(s) loaded")
        return df
    except Exception:
        print(f"  ⚠ {FAVORITES_TABLE} not found — run 02__favorites first. Skipping.")
        return pd.DataFrame(columns=["ticker", "company"])

In [0]:
raw_sources: dict[str, pd.DataFrame] = {}

if INGEST_SP500:
    print("Fetching S&P 500...")
    raw_sources["sp500"] = fetch_sp500()
    print(f"  ✓ {len(raw_sources['sp500'])} tickers")

if INGEST_R3000:
    print("Fetching Russell 3000...")
    raw_sources["r3000"] = fetch_russell3000()
    print(f"  ✓ {len(raw_sources['r3000'])} tickers")

print("Loading favorites...")
favorites_df = fetch_favorites()

## 2. Merge into unified ticker universe

In [0]:
master = favorites_df.copy()
master["is_favorite"] = True

for source_name in ["sp500", "r3000"]:
    flag_col = f"in_{source_name}"
    if source_name in raw_sources:
        idx_df = raw_sources[source_name][["ticker", "company"]].copy()
        idx_df[flag_col] = True
        master = master.merge(idx_df, on="ticker", how="outer", suffixes=("", f"_{source_name}"))
        if f"company_{source_name}" in master.columns:
            master["company"] = master["company"].fillna(master[f"company_{source_name}"])
            master.drop(columns=[f"company_{source_name}"], inplace=True)
    else:
        master[flag_col] = False

bool_cols = ["is_favorite", "in_sp500", "in_r3000"]
for col in bool_cols:
    if col not in master.columns:
        master[col] = False
    master[col] = master[col].fillna(False)

master = master[["ticker", "company"] + bool_cols].drop_duplicates("ticker").sort_values("ticker")

print(f"\nUnified universe: {len(master):,} unique tickers")
print(master[bool_cols].sum().to_string())

## 3. Write to Delta

In [0]:
schema = StructType([
    StructField("ticker",      StringType(),  False),
    StructField("company",     StringType(),  True),
    StructField("is_favorite", BooleanType(), False),
    StructField("in_sp500",    BooleanType(), False),
    StructField("in_r3000",    BooleanType(), False),
])

sdf = spark.createDataFrame(master, schema=schema)
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ticker")
    .saveAsTable(TARGET_TABLE)
)

print(f"✓ Written {sdf.count():,} tickers → {TARGET_TABLE}")

## 4. Sanity check

In [0]:
spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_tickers,
        SUM(CAST(is_favorite AS INT))                           AS favorites,
        SUM(CAST(in_sp500    AS INT))                           AS in_sp500,
        SUM(CAST(in_r3000    AS INT))                           AS in_r3000,
        SUM(CASE WHEN in_sp500 AND in_r3000 THEN 1 ELSE 0 END) AS in_both,
        SUM(CASE WHEN is_favorite AND NOT in_sp500
                  AND NOT in_r3000 THEN 1 ELSE 0 END)           AS favorites_only
    FROM {TARGET_TABLE}
""").show()

if spark.sql(f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE is_favorite").collect()[0][0] > 0:
    print("\nFavorite tickers:")
    spark.sql(f"""
        SELECT ticker, company, in_sp500, in_r3000
        FROM {TARGET_TABLE} WHERE is_favorite ORDER BY ticker
    """).show()